In [1]:
import re
import nltk
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv("Amazon-Review-Dataset.csv")

In [3]:
df.shape
df.head()

,Review,Sentiment
0,Fast shipping but this product is very cheaply...,1
1,This case takes so long to ship and it's not e...,1
2,Good for not droids. Not good for iPhones. You...,1
3,The cable was not compatible between my macboo...,1
4,The case is nice but did not have a glow light...,1


In [4]:
df[["Sentiment"]].value_counts()

Sentiment
1            5000
2            5000
3            5000
4            5000
5            5000
Name: count, dtype: int64

In [5]:
df.isnull().sum()

Review       1
Sentiment    0
dtype: int64

In [6]:
df = df.dropna()

## Target Variable Binarization

In [7]:
df["Sentiment"] = (df["Sentiment"] > 3).astype("int")

In [8]:
df[["Sentiment"]].value_counts()

Sentiment
0            15000
1             9999
Name: count, dtype: int64

# Text Preprocessing

## 1. Convert to Lower case

In [9]:
df["Review"] = df["Review"].str.lower()

## 2. Remove HTML Tags

In [10]:
def remove_html(text):
    return re.sub(r"<[^>]+>", " ", text)

df["Review"] = df["Review"].apply(remove_html)

## 3. Remove Punctuations

In [11]:
def remove_punct(text):
    return re.sub(r"[^a-z\s]", "", text)

df["Review"] = df["Review"].apply(remove_punct)

## 4. Remove StopWords

In [12]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
stop_words = stopwords.words("english")

def remove_stopwords(text):
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words]
    return " ".join(tokens)

df["Review"] = df["Review"].apply(remove_stopwords)

## 5. Stemming

In [14]:
ps = PorterStemmer()

def stemming(text):
    stemmed_tokens = []
    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_word = ps.stem(token)
        stemmed_tokens.append(stemmed_word)

    return " ".join(stemmed_tokens)

df["Review"] = df["Review"].apply(stemming)

## 6. Vectorization

In [15]:
tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["Review"])
y = df["Sentiment"]

X = X.toarray()

# Train-Test-Split

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Dataset and DataLoader

In [17]:
train_dataset = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train.values).float())
test_dataset = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test.values).float())

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Build our RNN LSTM

In [18]:
class LSTM(nn.Module):
    def __init__(self, input_size, num_layers=2, hidden_size=128, dropout_prob=0.3):
        super().__init__()

        self.num_layers=num_layers
        self.hidden_size=hidden_size

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_prob if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, (hn, cn) = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

# Model Training

In [19]:
input_size = X_train.shape[1]

model = LSTM(input_size)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters())

In [20]:
epochs = 10
for epoch in range(epochs):
    model.train()
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1)  # (batch_size, 1, seq_len)

        outputs = model(Xb)

        loss = criterion(outputs.squeeze(1), yb)
        loss.backward()
        optimizer.step()

    print(f"epoch {epoch + 1} / {epochs} and loss = {loss.item()}")

epoch 1 / 10 and loss = 0.33244413137435913
epoch 2 / 10 and loss = 0.4419676661491394
epoch 3 / 10 and loss = 0.4801591634750366
epoch 4 / 10 and loss = 0.15676505863666534
epoch 5 / 10 and loss = 0.34592726826667786
epoch 6 / 10 and loss = 0.2801288366317749
epoch 7 / 10 and loss = 0.23135797679424286
epoch 8 / 10 and loss = 0.10781028866767883
epoch 9 / 10 and loss = 0.2266790121793747
epoch 10 / 10 and loss = 0.24016238749027252


In [21]:
from sklearn.metrics import classification_report, confusion_matrix

all_preds = []
all_targets = []

model.eval()
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        preds = (torch.sigmoid(outputs.squeeze(1)) > 0.5).float()

        all_preds.extend(preds.numpy())
        all_targets.extend(yb.numpy())

print(classification_report(all_targets, all_preds))
print("Confusion Matrix:\n", confusion_matrix(all_targets, all_preds))

              precision    recall  f1-score   support

         0.0       0.82      0.84      0.83      3000
         1.0       0.75      0.72      0.74      2000

    accuracy                           0.79      5000
   macro avg       0.79      0.78      0.78      5000
weighted avg       0.79      0.79      0.79      5000

Confusion Matrix:
 [[2529  471]
 [ 559 1441]]


In [23]:
import joblib

# 1. Save the fitted TF-IDF Vectorizer
joblib.dump(tf, 'tfidf_vectorizer_amazon.joblib')

# 2. Save the trained PyTorch model weights
torch.save(model.state_dict(), 'rnn_amazon_model.pth')
print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!
